## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Mount drive to access pdf
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Enable CUDA with force CMAKE
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 191.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 275.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 228.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 300.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have numpy 2.3.1 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.1 which is incompatible.
torch 2.6.

* Enable CUDA support for GPU acceleration w/ FORCE_CMAKE=1 to force the use of CMake for building


In [4]:
# For installing the libraries & downloading models from HF Hub
!pip install  tiktoken==0.9.0 \
                pypdf==5.6.0 \
                langchain==0.3.0 \
                langchain-community \
                chromadb \
                sentence-transformers==4.1.0 \
                huggingface_hub==0.33.0 \
                numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a w

In [1]:
# Intall PyMuPDF library to work with PDF
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 89.7 MB/s eta 0:00:00


In [2]:
# Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

# Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
# Assign Hugging Face repository and file name
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

* Store the name of the Hugging Face repository where the pre-trained model is located and specific filename of the model file within the repository.

In [ ]:
# Download model and repo from Hugging Face
model_path = hf_hub_download(
    repo_id= model_name_or_path, #Mention the repo id
    filename=model_basename #Mention the model name
)

* It downloads the specified model file from the Hugging Face repository and stores its local path in the model_path variable.

In [ ]:
# Initialize Llama object
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


* Initializes a Llama object, a class from the llama_cpp library used for loading and running Llama models.

#### Response

In [ ]:
# Define function to fetch response from Llama
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text'] # Return first choice from the output

* Response encapsulates the process of generating a response from your loaded language model (llm) with different querries and parameters.

In [ ]:
# Call response to test the output
response("What treatment options are available for managing hypertension?")

'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity'

* Treatement options are provided accurately.
* Answer is truncated due to max tokens limit.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input) # Pass the user input

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition and suspicion: Septic patients may present with non-specific symptoms such as fever, chills, tachycardia, tachypnea, altered mental status, or lactic acidosis. It is essential to have a high index of suspicion for sepsis in patients with known or suspected infections.\n2. Initial assessment and res'

* It explains what sepsis is and protocol for managing it in ICU but fails to address the context directly in terms of the protocol.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2) # Pass the user input

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch that extends from the large intestine. The following are common symptoms of appendicitis:\n\n1. Abdominal pain: The pain typically starts around the navel area and then shifts to the lower right side of the abdomen. It may be mild at first but can quickly become severe and constant.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and discomfort.\n3. Nausea and vomiting: Vomiting is'

* It's detailing unnecessary information and not being precise to the question.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3) # Pass the user input

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, or eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

* Accurate answer.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4) # Pass the user input

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to brain tissue, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for TBIs:\n\n1. Emergency care: The first priority is to ensure the person's airway is clear, they are breathing, and their heart is beating normally. In severe cases, emergency surgery may be required to remove hematomas or other obstructions.\n2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as"

* Pretty decent answer for first part of the query, but need more parameters for the later.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5) # Pass the user input

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sl"

* Response is slightly diverted from the actual questions in the query, generic to the fracture but not specific to the leg injury during hiking.

### Query 6: What are the common symptoms and treatments for pulmonary embolism?

In [ ]:
user_input_6 = "What are the common symptoms and treatments for pulmonary embolism?"
response(user_input_6) # Pass the user input

Llama.generate: prefix-match hit


'\n\nPulmonary embolism is a serious condition that occurs when a blood clot or other foreign substance travels to the lungs and blocks one or more of the arteries supplying blood to the lungs. This can lead to decreased oxygen supply to the body, which can result in various symptoms. Some common symptoms of pulmonary embolism include:\n\n1. Shortness of breath: This is often the most prominent symptom of pulmonary embolism. Patients may feel like they are unable to get enough air or that they are breathing too hard.\n2. Chest pain:'

* Response doesn't maintain the context specific to common symptoms and treatements.

### Query 7: Can you provide the trade names of medications used for treating hypertension?

In [ ]:
user_input_7 = "Can you provide the trade names of medications used for treating hypertension?"
response(user_input_7) # Pass the user input

Llama.generate: prefix-match hit


'\n\nYes, there are numerous medications used for treating hypertension (high blood pressure), and their trade names can vary depending on the specific drug class and manufacturer. Here are some common classes of antihypertensive drugs and their trade names:\n\n1. Thiazide diuretics: HCTZ (Hydrochlorothiazide), Diuril (Chlorothiazide), Maxzide (Hydrochlorothiazide and amlodipine)\n2. Beta-blockers: Tenormin (Atenolol), Lopressor (Metop'

* Fairly grounded response sharing some of the trade names.

### Query 8: What are the first-line options and alternatives for managing rheumatoid arthritis?

In [ ]:
user_input_8 = "What are the first-line options and alternatives for managing rheumatoid arthritis?"
response(user_input_8) # Pass the user input

Llama.generate: prefix-match hit


'\n\nRheumatoid arthritis (RA) is a chronic inflammatory disease that primarily affects the joints, although it can also affect other organs. The primary goal of RA treatment is to reduce symptoms, prevent joint damage, and improve function and quality of life.\n\nFirst-line options for managing rheumatoid arthritis include:\n\n1. Disease-modifying antirheumatic drugs (DMARDs): DMARDs are a group of medications that can slow down or stop the progression of RA by reducing inflammation and preventing'

* It's responding with related information but not specific to the question.

### Query 9: What are the diagnostic steps for suspected endocrine disorders?

In [ ]:
user_input_9 = "What are the diagnostic steps for suspected endocrine disorders?"
response(user_input_9) # Pass the user input

Llama.generate: prefix-match hit


"\n\n1. Obtain a detailed history: Endocrine disorders can present with nonspecific symptoms, so it is important to obtain a thorough history that includes information about the patient's symptoms, duration, and any associated signs or symptoms. It is also important to ask about family history of endocrine disorders, medication use, and lifestyle factors such as diet and exercise habits.\n2. Perform a physical examination: Endocrine disorders can affect various systems in the body, so a thorough physical examination is essential. Pay particular attention to signs of hormonal imbalances, such as growth abnormalities, skin changes"

* Responding with lots of unnecessary information like history of the endocrine disorders instead of answering contextually.

### Query 10: Who discovered thyroid gland disease?

In [ ]:
user_input_10 = "Who discovered thyroid gland disease?"
response(user_input_10)

Llama.generate: prefix-match hit


"\n\nThe discovery of thyroid gland diseases is attributed to several pioneers in medicine, including Robert Graves and Paracelsus. However, it was Dr. Robert Graves who first described the clinical syndrome of hyperthyroidism (Graves' Disease) in detail in 1835. He also discovered that the thyroid gland was responsible for producing the thyroid hormones. Paracelsus, a Swiss physician from the 16th century, is believed to have been one of the first to describe goiter, an enlarged thyroid gland, which can be a sign"

* Model is hallucinating here as it's not clearly answering who discovered the disease and seemingly diverting from the context.

## Question Answering using LLM with Prompt Engineering

In [ ]:
# System prompt to instruct llm
system_prompt = "You are a helpful medical assistant. Answer the user's question based on the provided medical context only in full sentences."

* System prompt defines instructions for the language model, telling it to act as a helpful medical assistant and to answer questions based only on the provided medical context, in full sentences.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\nIn a critical care unit, managing sepsis involves several steps to ensure the best possible outcome for the patient. First and foremost, early recognition and prompt initiation of treatment are crucial. This includes administering broad-spectrum antibiotics as soon as possible based on the suspected infection source and obtaining blood cultures before antibiotic administration if feasible.\n\nSecondly, providing adequate fluid resuscitation is essential to maintain adequate tissue perfusion and organ function. Crystalloid solutions are typically used for initial resuscitation, with colloids or blood products considered in cases of refractory hypotension'

* It seeems grounded and provides relevant response, maybe with help of more tokens and creativity/randomness we can get more accurate response.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = system_prompt+"\n"+"What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The common symptoms for appendicitis include:\n1. Sudden pain in the lower right abdomen that may start as mild and then gradually worsens over time.\n2. Loss of appetite and feeling sick to your stomach.\n3. Fever, which can be low-grade at first but can rise as high as 101 degrees Fahrenheit (38.3 degrees Celsius) or higher.\n'

* We need to increase tokens to get details symptoms.
* It fails to answer later part of the question.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = system_prompt+"\n"+"What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other parts of the body. The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system.\n\nEffective treatments for addressing sudden patchy hair loss include:\n1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system response, allowing hair to grow back. They can be applied topically or taken orally."

* Response shares unnecessary information not just relevant to the treatments or solutions.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = system_prompt+"\n"+"What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

Llama.generate: prefix-match hit


'\n\nFor a person who has sustained a physical injury to brain tissue and is experiencing temporary or permanent impairment of brain function, there are several treatment options that may be recommended by healthcare professionals. These treatments aim to manage symptoms, promote recovery, and improve overall quality of life.\n\n1. Medications: Depending on the specific symptoms, medications may be prescribed to help manage various conditions such as seizures, pain, or depression. For instance, anticonvulsants may be used to prevent seizures, while analgesics can help alleviate pain.\n2. Rehabilitation therapy: Rehabil'

* Due to lack of parameter for k value or max_tokens, it doesn't provide the full information specific to the desired context.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = system_prompt+"\n"+"What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5)

Llama.generate: prefix-match hit


'\n\nFirst and foremost, it is essential to ensure the safety and stability of the injured person. If possible, try to keep them warm and comfortable while waiting for medical help to arrive. Do not attempt to move them unless their condition is life-threatening or they are in imminent danger. Once medical professionals have assessed the situation, they may apply a splint or cast to immobilize the fractured leg. Pain management will also be an essential part of the treatment plan, which may include over-the-counter pain relievers or prescription medication.\n\nDuring recovery, it is crucial to follow'

* System token is proven to be really useful to maintain the context while anwering it accurately.

### Query 6: What are the common symptoms and treatments for pulmonary embolism?

In [ ]:
user_input_6 = system_prompt+"\n"+"What are the common symptoms and treatments for pulmonary embolism?"
response(user_input_6)

Llama.generate: prefix-match hit


'\nA pulmonary embolism is a serious condition where a blood clot blocks an artery in the lungs, obstructing blood flow. Common symptoms include shortness of breath, chest pain or discomfort, rapid heart rate, and unexplained anxiety or fear. In severe cases, there may be coughing up blood, rapid breathing, sweating, and bluish skin due to lack of oxygen. Treatment for a pulmonary embolism typically involves anticoagulant medications to prevent the clot from growing larger or forming new clots. These medications help thin the blood and restore normal'

* Accurately responds to the question and quite to the point with useful context.

### Query 7: Can you provide the trade names of medications used for treating hypertension?

In [ ]:
user_input_7 = system_prompt+"\n"+"Can you provide the trade names of medications used for treating hypertension?"
response(user_input_7)

Llama.generate: prefix-match hit


'\nYes, there are several trade names for medications commonly used to treat hypertension. Some examples include:\n1. Lisinopril (Prinivil, Zestril) - an ACE inhibitor\n2. Hydrochlorothiazide (Microzide, HCTZ) - a thiazide diuretic\n3. Metoprolol (Lopressor, Toprol XL) - a beta blocker\n4. Losartan (Cozaar) - an ARB (angiotensin receptor blocker)\n5. Nifedip'

* Examples provided as answer to trade names.

### Query 8: What are the first-line options and alternatives for managing rheumatoid arthritis?

In [ ]:
user_input_8 = system_prompt+"\n"+"What are the first-line options and alternatives for managing rheumatoid arthritis?"
response(user_input_8)

Llama.generate: prefix-match hit


'\nFirst-line options for managing rheumatoid arthritis typically include disease-modifying antirheumatic drugs (DMARDs) such as methotrexate, leflunomide, or sulfasalazine. These medications can help reduce inflammation and slow down joint damage. Another first-line option is biologic DMARDs like adalimumab, etanercept, or infliximab, which are used for more severe cases or when other DMARDs have not been effective. Alternative treatments may include corticosteroids for short-term sympt'

* It is related but not grounded response.

### Query 9: What are the diagnostic steps for suspected endocrine disorders?

In [ ]:
user_input_9 = system_prompt+"\n"+"What are the diagnostic steps for suspected endocrine disorders?"
response(user_input_9)

Llama.generate: prefix-match hit


'\n\nWhen a patient presents with symptoms suggestive of an endocrine disorder, the diagnostic process typically involves several steps. First, a thorough medical history and physical examination are conducted to identify any specific signs or symptoms that may point towards a particular endocrine condition. For instance, unexplained weight loss, increased thirst and urination, fatigue, and muscle weakness could indicate diabetes mellitus, while excessive sweating, heat intolerance, and rapid heartbeat might suggest hyperthyroidism.\n\nOnce clinical suspicion is raised, laboratory tests are ordered to confirm the diagnosis. Blood tests are commonly used'

* It does answer the question but with a lot of unnecessary information regarding symptoms and examples.

#### Query 10: Who discovered thyroid gland disease?

In [ ]:
user_input_10 = system_prompt+"\n"+"Who discovered thyroid gland disease?"
response(user_input_10)

Llama.generate: prefix-match hit


'\nBased on the medical context provided, it is important to clarify that the discovery of specific thyroid gland diseases has been made by various researchers and scientists over several centuries. However, one significant milestone in the history of thyroid disease discovery was the identification of iodine deficiency as a cause of goiter (enlarged thyroid gland) in the late 1800s by Swiss chemist Jean-Baptiste Fouquet and French physician Jules Guérin. Later, in the early 1900s, German endocrinologist Friedrich Goltz and American physi'

* With system token, it eliminates hallucination and now it returns response close to the actual context of the question.

#### Query 11: Cause of patch

In [ ]:
user_input_0 = system_prompt+"\n"+"Causes of patch"
response(user_input_0)

Llama.generate: prefix-match hit


"y hair loss:\n1. Alopecia Areata: This is an autoimmune condition where the body attacks its own hair follicles, causing circular patches of hair loss. It can affect anyone, but it's more common in people with a family history of the condition or those with other autoimmune diseases.\n2. Telogen Effluvium: This type of hair loss occurs when a large number of hairs enter the resting phase (telogen) at the same time, causing an excessive shedding of hair. It can be caused by stress, hormonal imbalances, or"

* Clear example of hallucination without using RAG.
* It is returning patch keyword related information without context, instead it should ask more questions or avoid answering.

## Data Preparation for RAG

### Loading the Data

In [ ]:
manual_pdf_path = "/content/drive/MyDrive/medical_diagnosis_manual.pdf" # Define the file name

* Defining the manual path name.

In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path) # Call PDF loader

* Creating an instance of the PyMuPDFLoader, which is a class designed to load content from PDF files using the PyMuPDF library.

In [ ]:
manual = pdf_loader.load() # Load PDF manual

* Loaded medical manual to process data

### Data Overview

#### Checking the first 5 pages

In [ ]:
# Validate first 5 pages from the manual
for i in range(100, 105):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 101
Vitamin D is a prohormone with several active metabolites that act as hormones. Vitamin D is metabolized
by the liver to 25(OH)D, which is then converted by the kidneys to 1,25(OH)2D (1,25-
dihydroxycholecalciferol, calcitriol, or active vitamin D hormone). 25(OH)D, the major circulating form, has
some metabolic activity, but 1,25(OH)2D is the most metabolically active. The conversion to 1,25(OH)2D
is regulated by its own concentration, parathyroid hormone (PTH), and serum concentrations of Ca and
phosphate.
[
Table 4-6. Actions of Vitamin D and its Metabolites]
Vitamin D affects many organ systems (see Table 4-6), but mainly it increases Ca and phosphate
absorption from the intestine and promotes normal bone formation and mineralization. Vitamin D and
related analogs may be used to treat psoriasis, hypoparathyroidism, renal osteodystrophy, and possibly
leukemia and breast, prostate, and colon cancers; they may also be used for immunosuppression.
Vitamin D Deficiency 

* Inspecting the content of a few pages in loaded PDF to ensure the loading process worked correctly and to get a sense of the text we'll be working with.

#### Checking the number of pages

In [ ]:
# Check the length of the manual
len(manual)

4114

* Also verifying the length of the manual/PDF.

### Data Chunking

In [ ]:
# Initialize a RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512, # Define the chunk size
    chunk_overlap=20 # Define the chunk overlap
)

* The text of the PDF manual is split into smaller, manageable chunks.
* Each chunk will share 20 tokens with the chunk that comes before it and the chunk that comes after it, each with size 512. This overlap helps ensure that context isn't lost when a topic spans across chunk boundaries.

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter) # Load the  text splitter and divide document into chunks

* First loads the entire content of the PDF specified by manual_pdf_path. Then, it uses the provided text_splitter (configured with desired chunk_size and chunk_overlap) to divide the loaded text into smaller document chunks.

In [ ]:
len(document_chunks) # Check the total number of the document chunks

8492

* Total number of document chunks, the medical manula is divided into based on chunk_size and chunk_overlap

In [ ]:
document_chunks[1].page_content

'rimasukhadia39@gmail.com\n8FDRW14EB2\nThis file is meant for personal use by rimasukhadia39@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [ ]:
document_chunks[-3].page_content

"X\nXanthelasma 579\nXanthine oxidase deficiency 3026\nXanthines 2064\nXanthochromia 1595\nXanthogranulomatous pyelonephritis 2379\nXanthomas 634, 645, 895\nXanthomatosis, cerebrotendinous 894\nX chromosome 2599, 3373 (see also Genetic)\nabnormalities of 3002-3005, 3376-3377, 3377\ninactivation of 3002, 3378\n47,XXX syndrome 3005\n48,XXXX syndrome 3005\n49,XXXXX syndrome 3005\n47,XYY syndrome 3005\nXenograft 1126 (see also Transplantation)\nXeroderma (xerosis) 661\nXerophthalmia 594\nvitamin A deficiency and 34\nXerosis 636\nXerostomia 501, 513-516\nradiation-induced 488, 505\nin Sjogren's syndrome 304\nX-linked adrenoleukodystrophy 3024\nX-linked agammaglobulinemia 1092, 1097, 1107-1108\nX-linked lymphoproliferative syndrome 1092, 1097, 1108\nX-rays 3252 (see also Radiation; Radiation therapy; Radiography)\nXylene poisoning 3348\nXylitol 518\nXylose absorption test 155, 3494, 3500\nXylose breath test 156, 157\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nX\n4102\nrimasukhadi

* This accesses the third-to-last element in the list.

In [ ]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

* This accesses the last element in the list.

In [ ]:
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nrimasukhadia39@gmail.com\n8FDRW14EB2\nThis file is meant for personal use by rimasukhadia39@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

* This accesses the second-to-last element in the list.

As expected, there are some overlaps:  

- The sentence '*The Merck Manual of Diagnosis & Therapy, 19th Edition*' appears in both chunks.  
- If we increase the `chunk_overlap`, the overlapping length of the sentence will also increase.

### Embedding

In [ ]:
# Initialize embedding model
embedding_model = SentenceTransformerEmbeddings(model_name='All-MiniLM-L6-V2')

/tmp/ipython-input-45-3722349102.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='All-MiniLM-L6-V2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

* Using 'All-MiniLM-L6-V2' instead of 'thenlper/gte-large' to support T4 GPU instance

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content) # Query first document chunk from page content
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content) # Query second document chunk from page content

* Initialize to generate embeddings for the content of the first two document chunks

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1)) # Get the length of first embedding
len(embedding_1)==len(embedding_2) # Validating the length of two embeddings

Dimension of the embedding vector  384


True

In [ ]:
embedding_1,embedding_2 # Display both the Embeddings

([-0.06563830375671387,
  0.09500198066234589,
  0.009579895995557308,
  -0.037542734295129776,
  0.047290340065956116,
  -0.003079405054450035,
  0.09121417999267578,
  0.08489809930324554,
  0.015482899732887745,
  -0.030295707285404205,
  0.09139353036880493,
  -0.02542988210916519,
  0.027230173349380493,
  -0.017774205654859543,
  -0.09589557349681854,
  -0.10088874399662018,
  -0.03516123443841934,
  0.0025506634265184402,
  -0.08777239918708801,
  -0.005021215882152319,
  -0.02834237925708294,
  0.00410174997523427,
  0.008087044581770897,
  -0.00476181972771883,
  -0.025701412931084633,
  0.07857128977775574,
  -0.03525175154209137,
  0.07059279829263687,
  0.024542812258005142,
  -0.06207921728491783,
  0.036931224167346954,
  0.016890984028577805,
  0.03125252574682236,
  -0.016261553391814232,
  0.03364076837897301,
  0.016497183591127396,
  -0.08152931928634644,
  -0.04473140090703964,
  -0.011678777635097504,
  -0.03023468144237995,
  0.006202297285199165,
  -0.03581623360

* The embedding model provides a fixed-length vector for any number of chunks.  
* This is necessary because we want to compare them for similarity.

### Vector Database

In [ ]:
# Create directory for Medical vector database
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

* Create a directory to store vector database

In [ ]:
len(document_chunks) # Check the length of document chunks

8492

In [ ]:
# Initialize an empty Chroma vector store instance, explicitly setting the embedding function
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipython-input-50-1164814734.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


* Initializes Chroma database set up and ready to receive your document chunks, configured to use chosen embedding model.

In [ ]:
# Instantiate the embedding model

# Define a batch size smaller than the maximum allowed batch size (5461)
batch_size = 2000

# Add documents to the vector store in batches
for i in range(0, len(document_chunks), batch_size):
    batch = document_chunks[i:i + batch_size]
    vectorstore.add_documents(batch)

# The vectorstore is now populated in batches
print(f"Successfully added {len(document_chunks)} documents to the vector store.")

Successfully added 8492 documents to the vector store.


* It was not posible to use form_documents with such large manual dataset using this model. So, we neded to divide the document into batches/chunks.
* Instead it calls add_documnents of vectorscore or Chroma database and passes current batch of document chunks.
* And then create embeddings for each document in the batch and stores both document content and their embeddings in the database.

In [ ]:
vectorstore.embeddings # Return embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='All-MiniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

* The HuggingFaceEmbeddings using 'All-MiniLM-L6-V2' model has the embedding function associated with the Chroma vectorstore.
* This confirms that the vectorstore is set up to use this specific model for creating and comparing embeddings.

In [ ]:
# Pass a query to search for similarity in the medical databse and an appropriate k value
vectorstore.similarity_search("Health care Medical Medicine",k=3)

[Document(metadata={'file_path': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'author': '', 'format': 'PDF 1.7', 'page': 3659, 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'subject': '', 'keywords': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'source': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'creator': 'Atop CHM to PDF Converter', 'trapped': '', 'creationDate': 'D:20120615054440Z', 'modDate': 'D:20250614222801Z'}, page_content='Containing Health Care Costs\nConceptually, total health care costs can be contained or decreased only by some combination of the\nfollowing:\n• Decreasing use of health care services\n• Decreasing reimbursement for services that are used\n• Decreasing overhead (payor, provider, or both)\nSome strategies adversely affect access to care or outcomes; others may improve care. Evaluating\ndifferent strategies is difficult, partly because accurately measuring patient-centered healt

* From the retrieved chunks, we observe that all the chunks are related to the key terms ['Health care', 'Medical', 'Medicine'].

### Retriever

In [ ]:
# Convert Chroma to Retriever to use similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2} # Retrieve top 2 most similar document chunks
)

* Retriever here goes and find information for the language model.
* This tells the retriever to find documents based on how similar their embeddings are to the embedding of your query.
* Also, it tells the retriever to return the top 2 most similar document chunks it finds in the vectorstore for any given query.

In [ ]:
# Get relevant documents for the specified question
rel_docs = retriever.get_relevant_documents("What are the common symptoms and treatments for pulmonary embolism?") #Complete the code to pass the query
rel_docs

/tmp/ipython-input-26-1819602446.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What are the common symptoms and treatments for pulmonary embolism?") #Complete the code to pass the query


[Document(metadata={'keywords': '', 'source': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'file_path': '/content/drive/MyDrive/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'subject': '', 'modDate': 'D:20250614222801Z', 'format': 'PDF 1.7', 'creationDate': 'D:20120615054440Z', 'author': '', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'page': 2079, 'trapped': ''}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleuritic chest pain, cough, and, in severe cases, syncope or ca

- We can observe that the two relevant chunks contain the answer to the query.  
- If we increase the k value, there is a chance that we might find the answer in even more chunks.  
- This is a hyperparameter that we need to tune to get the best context.

In [ ]:
# Get output from llm
model_output = llm(
      "What are the common symptoms and treatments for pulmonary embolism?", # Pass the query
      max_tokens=128, # Pass the maximum number of tokens
      temperature=0, # Pass the temperature
    )

Llama.generate: prefix-match hit


* Asking the LLM to answer the question based purely on its internal training data, without the benefit of the relevant document chunks from your medical manual that the RAG system would normally provide to show the difference in the quality and specificity of answers when RAG is not used, compared to when it is.

In [ ]:
model_output['choices'][0]['text'] # Check first choice from the llm output

'\n\nPulmonary embolism (PE) is a serious condition that occurs when a blood clot or other foreign substance travels to the lungs and blocks one or more of the arteries supplying blood to the lungs. This can lead to decreased oxygen supply to the body, which can be life-threatening if not treated promptly.\n\nSymptoms of pulmonary embolism may include:\n\n1. Shortness of breath: This is often the most common symptom of PE. The person may feel like they are unable to get enough air or that they are breathing too hard'

* This is the way to extract the main generated text string from the output of this type of language model.
* LLM can sometimes return multiple possible responses, here we are selecting first choice from the list of potential outputs. In most cases, this is the primary or best response generated by the model.

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
qna_system_message = """
You are a helpful medical assistant. Answer the user's question based on the provided medical context only.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input in complete sentences.

If the answer is not found in the context, respond "I don't know".
"""

In [ ]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below based on the following medical context:
{context}

###Question
Answer the following question:
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input, top_k=20) # Pass the user input, limiting top_k to 20

Llama.generate: prefix-match hit


"Based on the context provided, the protocol for managing sepsis in a critical care unit includes the following steps:\n1. Monitoring: The patient's vital signs, including ECG, systolic, diastolic, and mean blood pressure, respiratory rate and depth, pulse oximetry, urine flow by indwelling bladder catheter, body temperature, and clinical status are closely monitored.\n2. Diagnosis: If sepsis or septic shock is suspected, cultures are obtained of blood and any other appropriate specimens.\n3. Treatment:"

* It's providing context specific response as opposed to earlier when it listed the cause of sepsis unnecessarily.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2) # Pass the user input

Llama.generate: prefix-match hit


"Based on the context provided, the common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left"

* Due to token limitation it doesn't seem to answer the later part, second question whether it's curable via medicine or not and the solution if it's not.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3) # Pass the user input

Llama.generate: prefix-match hit


'Based on the context provided, sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. The most common symptoms include bald spots on the scalp and beard, but any hairy area may be involved. Hair loss may affect most or all of the body (alopecia universalis).\nThe diagnosis of alopecia areata is typically made based on clinical presentation, microscopic hair examination, or scalp biopsy. Microscopic hair examination or scalp biopsy may differentiate'

* Question asks 2 different questions and it answers accurately both with respective contexts.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4) # Pass the user input

Llama.generate: prefix-match hit


'Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include:\n1. Supportive care: This includes preventing systemic complications due to immobilization such as pneumonia, UTI, and thromboembolic disease. Providing good nutrition and preventing pressure ulcers is also important. For patients whose coma exceeds 24 hours, a prolonged period of rehabilitation, particularly in cognitive and emotional areas, may be required.\n2. Physical therapy: To prevent limb contract'

* It recommends the treatments very precisely based on the desired context.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5) # Pass the user input

Llama.generate: prefix-match hit


"Based on the context provided, here's the answer:\nFirst, it is important to assess if there are any life-threatening injuries or signs of hemorrhagic shock. If so, the person should be treated accordingly in an emergency setting. For a fractured leg, the first step is usually immobilization with a splint to prevent further injury and decrease pain. The RICE (rest, ice, compression, elevation) method can also be used to help minimize swelling and pain. Pain is typically treated with opioids.\nDefinitive treatment for the fracture may involve reduction,"

* The answer is accurate without any unnecessary information.

### Query 6: Who discovered thyroid gland disease?

In [ ]:
user_input_6 = "Who discovered thyroid gland disease?"
generate_rag_response(user_input_6)

Llama.generate: prefix-match hit


"I don't know. The context does not provide information about who discovered thyroid gland disease."

* As expected from the system prompt, it is accurately returning the 'I don't know' response because the context to the question is not part of the entire manual.
* The model has successfully eliminated hallucinating.

### Fine-tuning

#### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input,temperature=0.5, max_tokens=150)

Llama.generate: prefix-match hit


"Based on the context provided, here's how you can answer the user's question:\nThe protocol for managing sepsis in a critical care unit includes several steps. First, if bacteremia, sepsis, or septic shock is suspected, cultures are obtained of blood and any other appropriate specimens. Early treatment with an appropriate antimicrobial regimen is given after cultures are obtained to improve survival. Antibiotics should be adjusted according to the results of culture and susceptibility testing, surgically draining any abscesses may be necessary, and internal devices that are the suspected source of bacteria should usually be removed. Patients in shock are critically ill and should be admitted to an IC"

* Response is definitely more creative and precise with the increase in temperature as opposed to focused and deterministic.
* There's not much increase in tokens but still you can see impact as it specifies the relatin to ICU.

#### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2, top_k=3, temperature=0.5, max_tokens=350)

Llama.generate: prefix-match hit


"Based on the context provided, the common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), an increase in pain from passive extension of the right hip joint that stretches the iliopsoas muscle (psoas sign), or pain caused by passive internal rotation of the flexed thigh (obturator sign). Low-grade fever (rectal temperature 37.7 to 38.3° C [100 to 101° F]) is common.\n\nAppendicitis cannot be cured via medicine alone, and surgical removal is the treatment of choice. The context describes that withou

* We can clearly difference with fine tuning the parameters like max_tokens and tmperature for creativity, it doesn't only answer the first half of the query/question but also the second half it missed earlier with high precision.
* It states accurately whether it's curable via medicine and reasoning in that context.

#### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3, top_p=0.98, top_k=20, max_tokens=256)

Llama.generate: prefix-match hit


'Based on the context provided, sudden patchy hair loss, also known as alopecia areata, is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers. The most common symptoms include bald spots on the scalp and beard, although any hairy area may be involved. Hair loss may affect most or all of the body (alopecia universalis).\nThe diagnosis of alopecia areata is typically made based on clinical presentation and microscopic examination of the hair shaft or scalp biopsy. Scalp biopsy may differentiate scarring from nonscarring forms, and immunofluorescence studies may help identify underlying disorders such as lupus erythematosus, lichen planopilaris, and systemic sclerosis.\nThe treatment for alopecia areata includes topical minoxidil (2% for women, 2% or 5% for men) which prolongs the anagen growth phase and gradually enlarges miniaturized follicles into mature terminal hairs. Finasteride, a medication that inhibits the 5α-reductase enzym

* It accurately describes the symptoms and treatements/solutions for patchy hair loss. However, it doesn't clearly state the causes of it.
* Tokens are certainly increased but due to top_p value, model will only consider tokens whose cumulative probability exceeds 95% and considers 20 most likely tokens.

#### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4, k=3, top_k=20, max_tokens=256)

Llama.generate: prefix-match hit


"Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include:\n1. Supportive care: This includes preventing systemic complications due to immobilization such as pneumonia, UTI, and thromboembolic disease. Providing good nutrition and preventing pressure ulcers is also important. For patients whose coma exceeds 24 hours, a prolonged period of rehabilitation, particularly in cognitive and emotional areas, may be required.\n2. Physical therapy: To prevent limb contractures.\n3. Support groups: May provide assistance to the families of brain-injured patients.\nThere is no specific treatment for brain damage, but these measures can help improve the patient's quality of life and manage any complications that may arise."

* Very precise response as before except little more detailed due to increase in max tokens parameter.

#### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5, k=3, top_k=20, max_tokens=300)

Llama.generate: prefix-match hit


"Based on the context provided, here's the answer:\n\nFor a person who has fractured their leg during a hiking trip, it is important to first assess if there are any life-threatening injuries or signs of hemorrhagic shock. If so, they should be treated accordingly in an emergency department. For the fractured leg, the first step would be to immobilize it immediately by splinting to prevent further injury and decrease pain. Pain can be managed with opioids. Definitive treatment for the fracture may involve reduction, which usually requires analgesia or sedation. Closed reduction is maintained by casting, while open reduction may require surgical hardware such as pins, screws, plates, or external fixators. RICE (rest, ice, compression, and elevation) can also be beneficial for soft-tissue injuries. Patients should avoid lifting heavy items, stooping, reaching, or jumping during recovery. Ambulation exercises are typically started after about 11 days, and patients should be taught to do d

* Model is working as expected giving very detailed and accurate response on each and every part of the question.
* It's sticking to the context with same value of k, 3 is moderate value to balance between not enough relevant context to noisy information.

### Query 6: Who discovered thyroid gland disease?

In [ ]:
user_input_6 = "Who discovered thyroid gland disease?"
generate_rag_response(user_input_6, k=3, top_k=20, max_tokens=250)

Llama.generate: prefix-match hit


'I don\'t know". The context does not provide information on who discovered thyroid gland disease.'

- Since the context provided doesn't help with the query, the model has responded correctly based on the prompt design.  

- However, technically n such cases there is a chance that it might not be present in the top k context. But here, it is not possible to increase due to token limitations.

## Output Evaluation

---



Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are an evaluator. Rate whether the answer is fully supported by the context. Use a score from 1 (not grounded) to 5 (fully grounded).
Explain briefly.
Use a score from 1 (not grounded) to 5 (fully grounded)
"""

relevance_rater_system_message = """
You are an evaluator. Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance. Use a score from 1 (irrelevant) to 5 (highly relevant).
Explain briefly.
Use a score from 1 (irrelevant) to 5 (highly relevant)
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=450)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5 (fully grounded). The context provides detailed information about the management of critically ill patients, including the monitoring and testing required for sepsis diagnosis and treatment. The answer accurately summarizes the steps involved in managing sepsis in a critical care unit, including obtaining cultures, administering empiric antibiotics, adjusting therapy based on culture results, and monitoring the patient's condition. The context also mentions the importance of ICU admission for patients with shock and the necessary monitoring parameters, which further supports the accuracy of the answer.

 Score: 5 (Highly Relevant)

The answer directly addresses the main aspects of managing sepsis in a critical care unit as outlined in the question. It covers the steps for diagnosis, which includes obtaining cultures and suspecting bacteremia, sepsis, or septic shock. The answer also discusses the importance of early antibiotic treatment and adjusting the

- It got a perfect score because the response is both grounded in the context and relevant to the query.
- This means that both the retrieval and augmentation parts are good.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",max_tokens=500)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5 (fully grounded). The user accurately summarized the common symptoms of appendicitis from the context, and they correctly stated that appendicitis cannot be cured via medicine alone and that surgical removal is the standard treatment. They also provided information on the types of surgical procedures and potential complications, all of which were discussed in the context.

 I would give the answer a score of 5 for relevance. The answer directly addresses the user's question by providing a clear explanation of the common symptoms of appendicitis and confirming that it cannot be cured via medicine alone. The answer also describes the standard surgical procedure for treating appendicitis, which is an appendectomy. Additionally, the answer provides information about the prognosis with early surgery versus complications. All important aspects of the question have been addressed in the answer.


- Likewise, this one got a perfect score for groundedness and relevance as well as it correctly answers  and sticks to the context.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",max_tokens=550)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5, as it is fully grounded in the context provided. The context discusses alopecia areata as a cause of sudden patchy hair loss and mentions various treatment options for this condition. The answer expands upon these points by listing specific treatments (topical corticosteroids, immunotherapy, minoxidil, JAK inhibitors, and hair transplantation) and explaining their mechanisms of action. Additionally, the answer correctly identifies the possible causes of alopecia areata as genetics and environmental triggers and describes the characteristic features of this condition (inflammation at the border of the bald patch and abnormalities in the hair shaft).

 I would give the answer a score of 5 for relevance. The answer directly addresses the main aspects of the question by providing information on the causes (autoimmune disorder, genetics, environmental triggers) and effective treatments (topical corticosteroids, immunotherapy, minoxidil, JAK inhibitors, hair 

- With increased tokens, it accurately answers both the parts of the questions.
- Very well explained with the reasonings for both the scores.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",max_tokens=500)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as 5 (fully grounded). The answer accurately summarizes the treatments recommended for a person with brain damage based on the information provided in the context, which includes supportive care and rehabilitation services. The answer also correctly notes that there is no specific treatment mentioned in the context to restore normal brain function.

 Based on the user's question and the provided context, I would rate the answer as highly relevant with a score of 5. The answer addresses all the important aspects of the question by mentioning supportive care as the recommended treatment for brain injury patients, which includes preventing systemic complications, providing good nutrition, preventing pressure ulcers, and planning for rehabilitation services if necessary. The answer also clarifies that there is no specific treatment mentioned in the context to restore normal brain function.


- Again a perfectly grounded and highly relevant answer.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",max_tokens=450)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5 (fully grounded). The context provided in the text discusses various injuries and treatments related to bones, arteries, nerves, and rehabilitation. The user's question specifically asks about the necessary precautions and treatment steps for someone who has fractured their leg during hiking. The answer directly addresses this question by discussing the importance of seeking medical attention in an emergency department, ruling out life-threatening conditions such as hemorrhagic shock or arterial injuries, immobilizing the affected limb with splinting or casting, applying RICE principles for healing and pain management, and undergoing rehabilitation to regain strength and

 I would rate the answer as highly relevant (5). The answer addresses all the important aspects of the question regarding necessary precautions and treatment steps for a person with a fractured leg during a hiking trip. It covers the evaluation process in the emergency department, immob

- Fully grouded and highly relevant despite fewer tokens.
- It maintains the context and the relevance as expected.

### Query 6: What are the common symptoms and treatments for pulmonary embolism?

In [ ]:
user_input = "What are the common symptoms and treatments for pulmonary embolism?"
ground,rel = generate_ground_relevance_response(user_input=user_input,max_tokens=450)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5 (fully grounded). The user's response accurately summarizes the common symptoms and treatments for pulmonary embolism as described in the context provided. Additionally, the user includes information about the prevalence and demographics of PE, which is also mentioned in the context.

 I would give the answer a score of 5 for relevance. The answer directly addresses the user's question by listing the common symptoms and treatments for pulmonary embolism as stated in the context. Additionally, it provides some additional information about the prevalence and demographics of the condition.


- Perfect scores as it explains how accurately the model answers the question.

### Query 7: Can you provide the trade names of medications used for treating hypertension?

In [ ]:
user_input = "Can you provide the trade names of medications used for treating hypertension?"
ground,rel = generate_ground_relevance_response(user_input=user_input,max_tokens=450)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 4 (well grounded). The user provided several examples of specific medications within each class of antihypertensives, but did not provide trade names for every medication mentioned in the context. However, the user's response accurately summarized the types of medications commonly used to treat hypertension based on the information given in the context.

 Score: 4 (Highly Relevant)

The answer provides a list of common trade names for various classes of medications used to treat hypertension, which directly addresses the user's question. Although the context mentions some specific medications by name, the answer goes beyond that and includes a range of trade names for each class. This comprehensive response ensures that all essential aspects of the question are covered.


- For groundedness and relevance, the response includes both the scores and the reason for the point deduction.  

### Query 8: What are the first-line options and alternatives for managing rheumatoid arthritis?

In [ ]:
user_input = "What are the first-line options and alternatives for managing rheumatoid arthritis?"
ground,rel = generate_ground_relevance_response(user_input=user_input,max_tokens=450)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5 (fully grounded). The context provides detailed information about rheumatoid arthritis, its causes, pathophysiology, diagnosis, and treatment options. The answer accurately summarizes the first-line options for managing RA, which include supportive measures, NSAIDs, and DMARDs, as well as alternative treatments and surgery in severe cases. The information in the context fully supports the answer.

 I would rate the answer as highly relevant (5). The answer clearly addresses the main aspects of managing rheumatoid arthritis, including first-line options such as supportive measures, NSAIDs, DMARDs, corticosteroids, and alternative therapies. It also mentions surgery as a last resort for severe cases.


* Very well addresses both the parts of this question.
* Perfect scores.

### Query 9: What are the diagnostic steps for suspected endocrine disorders?

In [ ]:
user_input = "What are the diagnostic steps for suspected endocrine disorders?"
ground,rel = generate_ground_relevance_response(user_input=user_input,max_tokens=450)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 I would rate the answer as a 5 (fully grounded). The context provided in the text outlines various diagnostic steps for suspected endocrine disorders, including measuring hormone levels, autoantibody titers, clinical diagnosis, lifelong follow-up, screening relatives, and treatment. The user's answer accurately summarizes these steps and expands upon them with additional details where necessary.

 Based on the context provided in the question and the answer, I would rate the relevance as a 5 (highly relevant). The answer covers all the important aspects mentioned in the context, including measuring hormone levels, autoantibody titers, clinical diagnosis, lifelong follow-up, screening relatives, and treatment for suspected endocrine disorders. The answer also expands on some of the details provided in the context, such as differentiating between types of multiple endocrine deficiencies and the importance of early intervention for chronic mucocutaneous candidiasis.


* It got perfect score because not only it delivers the highly relevant and perfectly grounded answer but also provides additional details specific to the context.

## Actionable Insights and Business Recommendations

* Since this is related to medical or healthcare industry, it's important to be vigilant about the accurate results before accepting it.

* Due to the clear distinction between just llm and model with RAG results, parameter tuning is crucial.

* Large models face issues in the embedding phase with such large data/manual. Select models accordingly.

* Vector database creation time increases with the number of pages in the PDF document and it might fail to store. So, might as well proactively use batching ot document chunking to run it faster and process the DB storage stage without any issues.

* Retrieval parameter k is critical as the answer can be spread across multiple contexts. Meaning randomness or creativity factor plays a critical role.

* chunk_overlap ensures coherence, especially when context spans across chunks.

* max_tokens depends on query complexity; higher values yield detailed responses, while simple queries result in concise outputs despite large token limits due to prompt design and zero temperature.

* Refine prompt design and temperature settings to control response, length and creativity.

* Simplify prompts for accurate output evaluation and concisely precise scores because groundedness_rater_system_message and relevance_rater_system_message experienced token limitation ue to Metric, Eval Criteria and Instructions overload.

* Continuously adjust RAG parameters based on specific use cases for optimal performance.

* Prioritize groundedness and relevance in evaluations to ensure reliable and contextually accurate outputs.

* Establish a feedback loop to fine-tune parameters, improving performance for diverse query types.

<font size=6 color='blue'>Power Ahead</font>
___